# nnUNetV2 Kidney Stone Segmentation — Training Notebook

**Dataset:** Dataset501_KidneyStones
**Task:** Stone segmentation
**Configuration:** 3D Full Resolution
**Environment:** Google Colab (GPU required)

---

## Instructions
1. Make sure `Dataset501_KidneyStones/` is already inside `MyDrive/1THESIS_AMM/` on Google Drive
2. Run cells sequentially — don't skip
3. After preprocessing, **always run the tar-save cell** before switching runtimes
4. After a runtime reset, run Sections 1–2 then the **tar-restore cell** to recover preprocessed data

## Section 1: Runtime & Environment Setup

### Cell 1.1 — Check GPU

In [1]:
!nvidia-smi

Fri May  8 01:09:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Cell 1.2 — Install nnUNetV2

In [2]:
!pip install nnunetv2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 16.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 107.1 MB/s eta 0:00:0

### Cell 1.3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 2: Dataset Configuration

### Cell 2.1 — Paths (edit if needed)

In [4]:
# ============================================================
# Dataset502_KidneyCyst — path configuration
# ============================================================

# Where the dataset folder lives on Drive (imagesTr, labelsTr, dataset.json)
DATASET_DRIVE_PATH = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyCyst"

# Where training results (checkpoints) are backed up on Drive
RESULTS_DRIVE_PATH = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset502_KidneyCyst"

# TAR backup path — used to persist preprocessed data across runtime resets
TAR_PREPROCESSED_PATH = TAR_PREPROCESSED_PATH = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed_502.tar"

print("Dataset path  :", DATASET_DRIVE_PATH)
print("Results path  :", RESULTS_DRIVE_PATH)
print("TAR backup    :", TAR_PREPROCESSED_PATH)

Dataset path  : /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyCyst
Results path  : /content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset502_KidneyCyst
TAR backup    : /content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed_502.tar


### Cell 2.2 — Set nnUNet Environment Variables

In [5]:
import os

# Raw data lives on Drive (read-only during training)
os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_raw"

# Preprocessed data: local SSD (fast I/O for training)
# After preprocessing, this gets tarred to Drive.
# On resume, restore the tar here.
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"

# Results: backed up to Drive after each fold
os.environ["nnUNet_results"]       = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_results"

for k in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    print(f"{k}: {os.environ[k]}")

nnUNet_raw: /content/drive/MyDrive/1THESIS_AMM/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/1THESIS_AMM/nnUNet_results


### Cell 2.3 — Copy Dataset from Drive → Local SSD

In [ ]:
import shutil
from pathlib import Path

# Create local nnUNet folder structure
!mkdir -p /content/nnUNet_raw /content/nnUNet_preprocessed /content/nnUNet_results

# Copy dataset501 into nnUNet_raw on local SSD
src = Path(DATASET_DRIVE_PATH)
dst = Path("/content/nnUNet_raw/Dataset502_KidneyCyst")

if dst.exists():
    print("Dataset already on local SSD — skipping copy.")
else:
    print("Copying dataset from Drive to local SSD...")
    shutil.copytree(src, dst)
    print("Done.")

# Verify
images = list((dst / "imagesTr").glob("*.nii.gz"))
labels = list((dst / "labelsTr").glob("*.nii.gz"))
print(f"Images : {len(images)}")
print(f"Labels : {len(labels)}")
print(f"Match  : {len(images) == len(labels)}")

Copying dataset from Drive to local SSD...


## Section 3: Dataset Integrity Check

Quick sanity check before expensive preprocessing.
Reads `dataset.json` and spot-checks 5 random label files.

In [ ]:
import json, random
import numpy as np
import nibabel as nib
from pathlib import Path

base = Path("/content/nnUNet_raw/Dataset502_KidneyCyst")

# 1. Load dataset.json
dataset_json = base / "dataset.json"
assert dataset_json.exists(), "dataset.json not found!"
with open(dataset_json) as f:
    meta = json.load(f)
print("Dataset JSON loaded.")
print("Labels       :", meta.get("labels", "NOT FOUND"))
print("Channel names:", meta.get("channel_names", meta.get("modality", "NOT FOUND")))

# Read valid label values from dataset.json (handles any schema)
raw_labels = meta.get("labels", {})
if isinstance(raw_labels, dict):
    valid_label_values = set(int(v) for v in raw_labels.values())
elif isinstance(raw_labels, list):
    valid_label_values = set(range(len(raw_labels)))
else:
    valid_label_values = {0, 1, 2}  # fallback
print("Valid label values:", sorted(valid_label_values))

# 2. Spot-check 5 random labels
lbl_files = list((base / "labelsTr").glob("*.nii.gz"))
sample = random.sample(lbl_files, min(5, len(lbl_files)))

issues = []
for p in sample:
    data = np.asanyarray(nib.load(p).dataobj)
    uniques = set(np.unique(data).astype(int))
    bad = uniques - valid_label_values
    if bad:
        issues.append((p.name, bad))
    print(f"  {p.name}: {sorted(uniques)}")

if issues:
    print("\nISSUES FOUND:")
    for name, vals in issues:
        print(f"  {name}: unexpected values {vals}")
else:
    print("\nAll spot-checks passed!")

Dataset JSON loaded.
Labels       : {'background': 0, 'cyst': 1}
Channel names: {'0': 'CT'}
Valid label values: [0, 1]
  neg_047.nii.gz: [np.int64(0)]
  cyst_024.nii.gz: [np.int64(0), np.int64(1)]
  neg_084.nii.gz: [np.int64(0)]
  neg_065.nii.gz: [np.int64(0)]
  cyst_145.nii.gz: [np.int64(0), np.int64(1)]

All spot-checks passed!


## Section 4: Planning & Preprocessing

nnUNet will extract the dataset fingerprint, compute normalization stats,
design the 3D U-Net architecture, and preprocess all training cases.

> **After this section completes, always run Cell 4.2 (tar-save) before switching runtimes.**

### Cell 4.0 — Disk Check

In [ ]:
!df -h /content

### Cell 4.1 — Plan & Preprocess

In [6]:
import os, subprocess
from pathlib import Path

# ── FIX: nnUNet_raw must point to LOCAL SSD where the dataset was copied ──
os.environ["nnUNet_raw"]          = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"

# Results stay on Drive (so checkpoints survive runtime resets)
os.environ["nnUNet_results"] = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_results"

# Verify Dataset501 is actually there
dataset_path = Path("/content/nnUNet_raw/Dataset502_KidneyCyst")
if dataset_path.exists():
    images = list((dataset_path / "imagesTr").glob("*.nii.gz"))
    labels = list((dataset_path / "labelsTr").glob("*.nii.gz"))
    print(f"✅ Dataset found: {len(images)} images, {len(labels)} labels")
else:
    print("❌ Dataset NOT found at", dataset_path)
    print("Re-run Cell 2.3 to copy from Drive first.")

print("\nActive env vars:")
for k in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    print(f"  {k} = {os.environ[k]}")

❌ Dataset NOT found at /content/nnUNet_raw/Dataset502_KidneyCyst
Re-run Cell 2.3 to copy from Drive first.

Active env vars:
  nnUNet_raw = /content/nnUNet_raw
  nnUNet_preprocessed = /content/nnUNet_preprocessed
  nnUNet_results = /content/drive/MyDrive/1THESIS_AMM/nnUNet_results


In [ ]:
!nnUNetv2_plan_and_preprocess -d 502 --verify_dataset_integrity -np 4

Fingerprint extraction...
Dataset502_KidneyCyst
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100% 290/290 [06:24<00:00,  1.33s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.94047833 0.80871094 0.84089844]. 
Current patch size: (np.int64(128), np.int64(128), np.int64(128)). 
Current median shape: [421.84466019 497.08737864 453.39805825]
Attempting to find 3d_lowres con

### Cell 4.2 — 💾 Save Preprocessed Data as TAR to Drive

Run this **immediately after preprocessing finishes** and before any runtime switch.
The tar file is written directly to Drive so it survives session resets.

In [ ]:
import subprocess
from pathlib import Path

preprocessed_local = Path("/content/nnUNet_preprocessed")
tar_path = Path(TAR_PREPROCESSED_PATH)

# Verify source exists and has content
if not preprocessed_local.exists():
    raise FileNotFoundError("Local preprocessed folder not found — did preprocessing complete?")

contents = list(preprocessed_local.rglob("*"))
print(f"Files to archive: {len(contents)}")

# Run tar — write directly to Drive
print(f"Creating tar at: {tar_path}")
print("This may take a few minutes...")
result = subprocess.run(
    ["tar", "-cf", str(tar_path), "-C", "/content", "nnUNet_preprocessed"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("TAR FAILED:", result.stderr)
    raise RuntimeError("tar creation failed")

size_gb = tar_path.stat().st_size / 1e9
print(f"\nTAR saved successfully: {tar_path}")
print(f"Size: {size_gb:.2f} GB")
print("You can safely switch runtimes now.")

Files to archive: 2908
Creating tar at: /content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed_502.tar
This may take a few minutes...

TAR saved successfully: /content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed_502.tar
Size: 71.00 GB
You can safely switch runtimes now.


In [ ]:
from pathlib import Path

tar_path = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed_502.tar")

print("Exists:", tar_path.exists())
if tar_path.exists():
    print("Size GB:", tar_path.stat().st_size / (1024**3))
    print("Path:", tar_path)

Exists: True
Size GB: 66.12030982971191
Path: /content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed_502.tar


### Cell 4.3 — 📦 Restore Preprocessed Data from TAR (after runtime reset)

Run this after a runtime reset **instead of re-running preprocessing**.
Make sure Sections 1–2 (install + mount + env vars) are run first.

In [7]:
from pathlib import Path
import subprocess

tar_src = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed.tar")

print(f"Size: {tar_src.stat().st_size / 1e9:.2f} GB")

# Check actual file type (ignore the extension)
result = subprocess.run(["file", str(tar_src)], capture_output=True, text=True)
print(result.stdout)

# Peek at first few bytes
with open(tar_src, "rb") as f:
    print("Magic bytes:", f.read(8).hex())

Size: 71.00 GB
/content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed.tar: POSIX tar archive (GNU)

Magic bytes: 6e6e554e65745f70


In [15]:
import tarfile, os
from pathlib import Path

tar_src = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_preprocessed.tar")
dst = Path("/content/nnUNet_preprocessed")

if dst.exists():
    print("Already exists locally — skipping extraction.")
else:
    print("Extracting preprocessed tar to local SSD...")
    with tarfile.open(tar_src, "r") as tf:
        tf.extractall("/content")
    print("Done.")

os.environ["nnUNet_preprocessed"] = str(dst)

# Verify
dataset = dst / "Dataset502_KidneyCyst"
files = list(dataset.rglob("*"))
plans = dataset / "nnUNetPlans.json"
print(f"Files: {len(files)}")
print(f"nnUNetPlans.json: {plans.exists()}")
print(f"nnUNet_preprocessed → {os.environ['nnUNet_preprocessed']}")

Extracting preprocessed tar to local SSD...


/tmp/ipykernel_1016/3031851794.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall("/content")


Done.
Files: 2912
nnUNetPlans.json: True
nnUNet_preprocessed → /content/nnUNet_preprocessed


### Cell 4.4 — (Optional) Patch Batch Size in Plans

In [16]:
import json, shutil
from pathlib import Path

plans_path = Path("/content/nnUNet_preprocessed/Dataset502_KidneyCyst/nnUNetPlans.json")
backup_path = plans_path.with_suffix(".json.backup")

if not backup_path.exists():
    shutil.copy2(plans_path, backup_path)
    print("Backup created:", backup_path)

with open(plans_path) as f:
    plans = json.load(f)

old = plans["configurations"]["3d_fullres"]["batch_size"]
print("Current batch_size:", old)

# Uncomment to change:
plans["configurations"]["3d_fullres"]["batch_size"] = 2
with open(plans_path, "w") as f:
     json.dump(plans, f, indent=4)
print("Updated to:", 2)

# Display 3d_fullres config
print("\n=== 3d_fullres config ===")
print(json.dumps(plans["configurations"]["3d_fullres"], indent=2))

Backup created: /content/nnUNet_preprocessed/Dataset502_KidneyCyst/nnUNetPlans.json.backup
Current batch_size: 2
Updated to: 2

=== 3d_fullres config ===
{
  "data_identifier": "nnUNetPlans_3d_fullres",
  "preprocessor_name": "DefaultPreprocessor",
  "batch_size": 2,
  "patch_size": [
    128,
    128,
    128
  ],
  "median_image_size_in_voxels": [
    434.5,
    512.0,
    467.0
  ],
  "spacing": [
    0.9130857586860657,
    0.78515625,
    0.81640625
  ],
  "normalization_schemes": [
    "CTNormalization"
  ],
  "use_mask_for_norm": [
    false
  ],
  "resampling_fn_data": "resample_data_or_seg_to_shape",
  "resampling_fn_seg": "resample_data_or_seg_to_shape",
  "resampling_fn_data_kwargs": {
    "is_seg": false,
    "order": 3,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_seg_kwargs": {
    "is_seg": true,
    "order": 1,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_probabilities": "resample_data_or_seg_to_shape",
  "resampling_fn_pr

In [17]:
from pathlib import Path
from collections import Counter

folder = Path("/content/nnUNet_preprocessed/Dataset502_KidneyCyst/nnUNetPlans_3d_fullres")

print(Counter(p.suffix for p in folder.iterdir() if p.is_file()))

print("\nSample files:")
for p in sorted(folder.iterdir())[:30]:
    print(p.name)

Counter({'.b2nd': 580, '.pkl': 290, '.ini': 1})

Sample files:
cyst_000.b2nd
cyst_000.pkl
cyst_000_seg.b2nd
cyst_001.b2nd
cyst_001.pkl
cyst_001_seg.b2nd
cyst_002.b2nd
cyst_002.pkl
cyst_002_seg.b2nd
cyst_003.b2nd
cyst_003.pkl
cyst_003_seg.b2nd
cyst_004.b2nd
cyst_004.pkl
cyst_004_seg.b2nd
cyst_005.b2nd
cyst_005.pkl
cyst_005_seg.b2nd
cyst_006.b2nd
cyst_006.pkl
cyst_006_seg.b2nd
cyst_007.b2nd
cyst_007.pkl
cyst_007_seg.b2nd
cyst_008.b2nd
cyst_008.pkl
cyst_008_seg.b2nd
cyst_009.b2nd
cyst_009.pkl
cyst_009_seg.b2nd


In [18]:
from pathlib import Path

folder = Path("/content/nnUNet_preprocessed/Dataset502_KidneyCyst/nnUNetPlans_3d_fullres")

for f in folder.glob("*.ini"):
    print("Deleting:", f)
    f.unlink()

print("Done. Remaining .ini files:", list(folder.glob("*.ini")))

Deleting: /content/nnUNet_preprocessed/Dataset502_KidneyCyst/nnUNetPlans_3d_fullres/desktop.ini
Done. Remaining .ini files: []


## Section 5: Model Training (3D Full-Res)

Train all 5 folds. After each fold, checkpoints are on Drive under `nnUNet_results/`.

| GPU | Time per fold (approx) |
|-----|------------------------|
| T4  | 4–6 h |
| L4  | 2–3 h |
| A100 | 1–2 h |

> **Resume behaviour:** if a fold was interrupted, add `--c` to resume from last checkpoint.

### Cell 5.1 — Train Fold 0

In [ ]:
#!nnUNetv2_train 502 3d_fullres 0 --npz

Streaming output truncated to the last 5000 lines.
2026-05-06 06:34:37.595206: train_loss -0.363
2026-05-06 06:34:37.599110: val_loss -0.326
2026-05-06 06:34:37.602504: Pseudo dice [np.float32(0.5622)]
2026-05-06 06:34:37.605798: Epoch time: 28.56 s
2026-05-06 06:34:39.089414: 
2026-05-06 06:34:39.092559: Epoch 308
2026-05-06 06:34:39.095865: Current learning rate: 0.00718
2026-05-06 06:35:08.223958: train_loss -0.3622
2026-05-06 06:35:08.228209: val_loss -0.4615
2026-05-06 06:35:08.231390: Pseudo dice [np.float32(0.8179)]
2026-05-06 06:35:08.241014: Epoch time: 29.14 s
2026-05-06 06:35:09.714842: 
2026-05-06 06:35:09.718087: Epoch 309
2026-05-06 06:35:09.721221: Current learning rate: 0.00717
2026-05-06 06:35:39.530071: train_loss -0.3451
2026-05-06 06:35:39.535945: val_loss -0.3166
2026-05-06 06:35:39.545333: Pseudo dice [np.float32(0.7515)]
2026-05-06 06:35:39.549109: Epoch time: 29.82 s
2026-05-06 06:35:41.053853: 
2026-05-06 06:35:41.057240: Epoch 310
2026-05-06 06:35:41.060452: C

### Cell 5.2 — Train Fold 1

In [ ]:
#!nnUNetv2_train 502 3d_fullres 1 --npz

Streaming output truncated to the last 5000 lines.
2026-05-06 15:44:45.993300: val_loss -0.2293
2026-05-06 15:44:46.001250: Pseudo dice [np.float32(0.5652)]
2026-05-06 15:44:46.006043: Epoch time: 28.83 s
2026-05-06 15:44:47.542415: 
2026-05-06 15:44:47.545614: Epoch 305
2026-05-06 15:44:47.548779: Current learning rate: 0.00721
2026-05-06 15:45:16.019264: train_loss -0.3536
2026-05-06 15:45:16.024267: val_loss -0.3209
2026-05-06 15:45:16.035732: Pseudo dice [np.float32(0.5627)]
2026-05-06 15:45:16.042263: Epoch time: 28.48 s
2026-05-06 15:45:17.502131: 
2026-05-06 15:45:17.505572: Epoch 306
2026-05-06 15:45:17.508862: Current learning rate: 0.0072
2026-05-06 15:45:45.915256: train_loss -0.3952
2026-05-06 15:45:45.921073: val_loss -0.2408
2026-05-06 15:45:45.925575: Pseudo dice [np.float32(0.5856)]
2026-05-06 15:45:45.929819: Epoch time: 28.41 s
2026-05-06 15:45:47.447674: 
2026-05-06 15:45:47.450657: Epoch 307
2026-05-06 15:45:47.453805: Current learning rate: 0.00719
2026-05-06 15:46

### Cell 5.3 — Train Fold 2

In [ ]:
!nnUNetv2_train 502 3d_fullres 2 --npz --c

Streaming output truncated to the last 5000 lines.
2026-05-07 05:22:53.422583: Current learning rate: 0.00721
2026-05-07 05:23:22.379310: train_loss -0.3958
2026-05-07 05:23:22.384284: val_loss -0.3262
2026-05-07 05:23:22.388609: Pseudo dice [np.float32(0.447)]
2026-05-07 05:23:22.396157: Epoch time: 28.96 s
2026-05-07 05:23:23.898366: 
2026-05-07 05:23:23.901413: Epoch 306
2026-05-07 05:23:23.904331: Current learning rate: 0.0072
2026-05-07 05:23:51.163071: train_loss -0.294
2026-05-07 05:23:51.168842: val_loss -0.2885
2026-05-07 05:23:51.173979: Pseudo dice [np.float32(0.6767)]
2026-05-07 05:23:51.178567: Epoch time: 27.27 s
2026-05-07 05:23:52.713080: 
2026-05-07 05:23:52.716246: Epoch 307
2026-05-07 05:23:52.719585: Current learning rate: 0.00719
2026-05-07 05:24:20.031023: train_loss -0.3223
2026-05-07 05:24:20.040117: val_loss -0.3532
2026-05-07 05:24:20.047076: Pseudo dice [np.float32(0.5486)]
2026-05-07 05:24:20.055902: Epoch time: 27.32 s
2026-05-07 05:24:21.604457: 
2026-05-0

### Cell 5.4 — Train Fold 3

In [12]:
!nnUNetv2_train 502 3d_fullres 3 --npz

Streaming output truncated to the last 5000 lines.
2026-05-07 14:53:51.869337: train_loss -0.3681
2026-05-07 14:53:51.874007: val_loss -0.3081
2026-05-07 14:53:51.877689: Pseudo dice [np.float32(0.6409)]
2026-05-07 14:53:51.882283: Epoch time: 28.37 s
2026-05-07 14:53:53.362405: 
2026-05-07 14:53:53.365649: Epoch 307
2026-05-07 14:53:53.368573: Current learning rate: 0.00719
2026-05-07 14:54:22.485935: train_loss -0.364
2026-05-07 14:54:22.490986: val_loss -0.1892
2026-05-07 14:54:22.500108: Pseudo dice [np.float32(0.4249)]
2026-05-07 14:54:22.503148: Epoch time: 29.13 s
2026-05-07 14:54:24.003976: 
2026-05-07 14:54:24.016513: Epoch 308
2026-05-07 14:54:24.019791: Current learning rate: 0.00718
2026-05-07 14:54:52.006734: train_loss -0.3969
2026-05-07 14:54:52.012231: val_loss -0.2085
2026-05-07 14:54:52.021595: Pseudo dice [np.float32(0.6584)]
2026-05-07 14:54:52.026641: Epoch time: 28.0 s
2026-05-07 14:54:53.535639: 
2026-05-07 14:54:53.538600: Epoch 309
2026-05-07 14:54:53.541723: C

### Cell 5.5 — Train Fold 4

In [19]:
!nnUNetv2_train 502 3d_fullres 4 --npz

Streaming output truncated to the last 5000 lines.
2026-05-08 04:03:24.841151: Pseudo dice [np.float32(0.8532)]
2026-05-08 04:03:24.846139: Epoch time: 27.19 s
2026-05-08 04:03:26.387623: 
2026-05-08 04:03:26.390563: Epoch 309
2026-05-08 04:03:26.393320: Current learning rate: 0.00717
2026-05-08 04:03:54.227262: train_loss -0.3263
2026-05-08 04:03:54.232813: val_loss -0.3193
2026-05-08 04:03:54.240048: Pseudo dice [np.float32(0.5037)]
2026-05-08 04:03:54.246332: Epoch time: 27.84 s
2026-05-08 04:03:55.785307: 
2026-05-08 04:03:55.788344: Epoch 310
2026-05-08 04:03:55.791353: Current learning rate: 0.00716
2026-05-08 04:04:22.995680: train_loss -0.3538
2026-05-08 04:04:23.005441: val_loss -0.3317
2026-05-08 04:04:23.010291: Pseudo dice [np.float32(0.7939)]
2026-05-08 04:04:23.017296: Epoch time: 27.21 s
2026-05-08 04:04:24.524712: 
2026-05-08 04:04:24.528024: Epoch 311
2026-05-08 04:04:24.539080: Current learning rate: 0.00715
2026-05-08 04:04:51.322586: train_loss -0.3345
2026-05-08 04

### Cell 5.6 — Manual Resume (specific fold)

Use this if a fold was interrupted and you need to resume it specifically.

In [ ]:
# Uncomment the fold you need to resume:

# !nnUNetv2_train 501 3d_fullres 0 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 1 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 2 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 3 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 4 --c --npz --num_epochs 200

# Validation only (run after fold completes):
# !nnUNetv2_train 501 3d_fullres 0 --val --npz

## Section 6: Find Best Configuration

After all 5 folds complete, find the best configuration and postprocessing.

In [ ]:
import os
os.environ["nnUNet_results"] = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_results"
!ls -lah /content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset502_KidneyCyst

In [ ]:
!nnUNetv2_find_best_configuration 501 \
  -c 3d_fullres \
  -tr nnUNetTrainer \
  -p nnUNetPlans \
  --disable_ensembling

In [ ]:
!cat /content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones/inference_instructions.txt

## Section 7: 5-Fold Cross-Validation Evaluation

Generate per-fold validation predictions and compute per-class metrics.
Produces `cv_summary.json` for your thesis.

### Cell 7.1 — Generate Validation Predictions for All Folds

In [ ]:
import os, json, subprocess, shutil
from pathlib import Path

dataset_id = 501
config = "3d_fullres"
preprocessed_dir = Path(f"/content/nnUNet_preprocessed/Dataset{dataset_id}_KidneyStones")
raw_dir = Path(f"/content/nnUNet_raw/Dataset{dataset_id}_KidneyStones")
results_dir = Path(f"/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset{dataset_id}_KidneyStones")

# Auto-detect trainer folder
trainer_candidates = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
assert len(trainer_candidates) > 0, "No trained model found! Complete Section 5 first."
trainer_dir = trainer_candidates[0]
print("Trainer folder:", trainer_dir.name)

# Load splits
splits_file = preprocessed_dir / "splits_final.json"
assert splits_file.exists(), f"{splits_file} not found!"
with open(splits_file) as f:
    splits = json.load(f)

cv_base = Path("/content/cv_predictions")
cv_base.mkdir(exist_ok=True)

for fold in range(5):
    print(f"\n=== Fold {fold} ===")
    val_cases = splits[fold]['val']
    print(f"Validation cases: {len(val_cases)}")

    fold_in  = cv_base / f"fold_{fold}_input"
    fold_out = cv_base / f"fold_{fold}_output"
    fold_in.mkdir(exist_ok=True)
    fold_out.mkdir(exist_ok=True)

    # Clean old symlinks
    for p in list(fold_in.iterdir()):
        p.unlink()

    missing = 0
    for case in val_cases:
        src = raw_dir / "imagesTr" / f"{case}_0000.nii.gz"
        dst = fold_in / f"{case}_0000.nii.gz"
        if src.exists():
            if dst.exists(): dst.unlink()
            dst.symlink_to(src)
        else:
            missing += 1
            print(f"  MISSING: {src.name}")
    if missing:
        print(f"  WARNING: {missing} cases missing.")

    # Run prediction for this fold
    cmd = [
        "nnUNetv2_predict",
        "-i", str(fold_in),
        "-o", str(fold_out),
        "-d", str(dataset_id),
        "-c", config,
        "-f", str(fold),
        "-chk", "checkpoint_best.pth",
        "-npp", "1",
        "-nps", "1",
        "--verbose"
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR!")
        print(result.stdout[-1000:])
        print(result.stderr[-1000:])
    else:
        print("Prediction complete.")

    # Merge into single folder
    merged = cv_base / "merged"
    merged.mkdir(exist_ok=True)
    for pred in fold_out.glob("*.nii.gz"):
        shutil.copy2(pred, merged / pred.name)

print(f"\nAll predictions merged into: {cv_base / 'merged'}")
print(f"Total files: {len(list((cv_base / 'merged').glob('*.nii.gz')))}")

### Cell 7.2 — Compute Per-Class Metrics & Save summary.json

In [ ]:
!pip install medpy scikit-learn -q

In [ ]:
import json, numpy as np, nibabel as nib
from pathlib import Path
from tqdm.notebook import tqdm
from medpy.metric.binary import hd95
from sklearn.metrics import precision_score, recall_score, f1_score

pred_dir   = Path("/content/cv_predictions/merged")
label_dir  = Path("/content/nnUNet_raw/Dataset501_KidneyStones/labelsTr")
output_json = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones/cv_summary.json")
output_json.parent.mkdir(parents=True, exist_ok=True)

# ── Read class names from dataset.json ────────────────────────────────
with open("/content/nnUNet_raw/Dataset501_KidneyStones/dataset.json") as f:
    meta = json.load(f)
raw_labels = meta.get("labels", {})
if isinstance(raw_labels, dict):
    # nnUNet v2 format: {"background": 0, "stone": 1, ...}
    class_names = {int(v): k for k, v in raw_labels.items() if int(v) != 0}
else:
    class_names = {i: f"class_{i}" for i in range(1, len(raw_labels))}
classes = sorted(class_names.keys())
print("Evaluating classes:", class_names)

results = {c: {"DSC": [], "IoU": [], "HD95": [], "Precision": [], "Recall": [], "F1": []} for c in classes}
per_case_results = []

pred_files = sorted(pred_dir.glob("*.nii.gz"))
print(f"Found {len(pred_files)} predictions.")

for pf in tqdm(pred_files):
    case_id = pf.name.replace(".nii.gz", "")
    lf = label_dir / f"{case_id}.nii.gz"
    if not lf.exists():
        print(f"Skipping {case_id}: label not found.")
        continue

    pred  = np.asanyarray(nib.load(pf).dataobj).astype(np.uint8)
    label = np.asanyarray(nib.load(lf).dataobj).astype(np.uint8)
    case_entry = {"case_id": case_id}

    for c in classes:
        p = (pred  == c).astype(np.uint8)
        l = (label == c).astype(np.uint8)

        if l.sum() == 0 and p.sum() == 0:
            dsc, iou, precision, recall, f1 = 1.0, 1.0, 1.0, 1.0, 1.0
            hd = None
        elif l.sum() == 0 or p.sum() == 0:
            dsc, iou, precision, recall, f1 = 0.0, 0.0, 0.0, 0.0, 0.0
            hd = None
        else:
            inter = np.logical_and(p, l).sum()
            union = np.logical_or(p, l).sum()
            dsc   = float(2.0 * inter / (p.sum() + l.sum()))
            iou   = float(inter / union)
            p_flat, l_flat = p.flatten(), l.flatten()
            precision = float(precision_score(l_flat, p_flat, zero_division=0))
            recall    = float(recall_score(l_flat, p_flat, zero_division=0))
            f1        = float(f1_score(l_flat, p_flat, zero_division=0))
            try:    hd = float(hd95(p, l))
            except: hd = None

        for metric, val in [("DSC", dsc), ("IoU", iou), ("HD95", hd),
                             ("Precision", precision), ("Recall", recall), ("F1", f1)]:
            results[c][metric].append(val)
        case_entry[class_names[c]] = {
            "DSC": dsc, "IoU": iou, "HD95": hd,
            "Precision": precision, "Recall": recall, "F1": f1
        }
    per_case_results.append(case_entry)

# Aggregate
summary = {"per_case": per_case_results, "aggregate": {}}
for c in classes:
    agg = {}
    for metric, vals in results[c].items():
        clean = [v for v in vals if v is not None]
        agg[f"{metric}_mean"] = float(np.mean(clean)) if clean else None
        agg[f"{metric}_std"]  = float(np.std(clean))  if clean else None
    summary["aggregate"][class_names[c]] = agg

with open(output_json, "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to:", output_json)
print("\nAggregate Metrics:")
for cls_name, metrics in summary["aggregate"].items():
    print(f"  {cls_name}:")
    for k, v in metrics.items():
        print(f"    {k}: {v:.4f}" if v is not None else f"    {k}: N/A")

## Section 8: Custom Inference with Profiling

5-fold ensemble inference on any volume(s) with per-volume latency and peak VRAM logging.

In [ ]:
import torch, json, numpy as np
from pathlib import Path
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO

# ── USER CONFIG ───────────────────────────────────────────────────────
input_path     = Path("/content/nnUNet_raw/Dataset501_KidneyStones/imagesTr")
output_folder  = Path("/content/inference_output_501")
checkpoint_name = "checkpoint_best.pth"
# ─────────────────────────────────────────────────────────────────────

output_folder.mkdir(exist_ok=True)

results_dir = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones")
candidates  = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
if not candidates:
    raise RuntimeError("No trained model found!")
model_folder = candidates[0]
print("Model folder:", model_folder.name)

predictor = nnUNetPredictor(
    tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
    device=torch.device('cuda', 0), verbose=False, allow_tqdm=True
)
predictor.initialize_from_trained_model_folder(
    str(model_folder), use_folds=(0, 1, 2, 3, 4), checkpoint_name=checkpoint_name
)
print("Predictor ready.")

files = [input_path] if input_path.is_file() else sorted(input_path.glob("*.nii.gz"))
if not files: raise ValueError(f"No .nii.gz files in {input_path}")

io = SimpleITKIO()
profile = []

for f in files:
    print(f"Processing: {f.name}")
    image, props = io.read_images([str(f)])
    torch.cuda.reset_peak_memory_stats()
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    ret = predictor.predict_single_npy_array(image, props, None, None, False)
    seg = ret[0] if isinstance(ret, tuple) else ret
    e.record(); torch.cuda.synchronize()
    latency = s.elapsed_time(e) / 1000
    vram    = torch.cuda.max_memory_allocated() / 1e9
    out_name = f.name.replace("_0000.nii.gz", ".nii.gz")
    io.write_seg(seg.astype(np.uint8), str(output_folder / out_name), props)
    entry = {"file": f.name, "latency_seconds": round(latency, 4), "peak_vram_gb": round(vram, 4)}
    profile.append(entry)
    print(f"  Latency: {latency:.3f}s  |  Peak VRAM: {vram:.2f} GB")

with open(output_folder / "inference_profile.json", "w") as f:
    json.dump(profile, f, indent=2)
print(f"\nDone. Profile saved to {output_folder}/inference_profile.json")

## Section 9: Save Results to Google Drive

Back up all training outputs (checkpoints, plans, CV predictions) to Drive.

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/nnUNet_results")
dst = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results")
dst.mkdir(parents=True, exist_ok=True)

print("Syncing nnUNet_results to Drive...")
for item in src.iterdir():
    dest_item = dst / item.name
    if item.is_dir():
        if dest_item.exists(): shutil.rmtree(dest_item)
        shutil.copytree(item, dest_item)
    else:
        shutil.copy2(item, dest_item)
    print(f"  Copied: {item.name}")
print("Done.")

## Section 10: Export Model for Deployment

Create a clean export folder with only the files needed for Streamlit/inference.

In [ ]:
from pathlib import Path
import shutil, json

dataset_raw  = Path("/content/nnUNet_raw/Dataset501_KidneyStones")
results_dir  = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones")
export_dir   = Path("/content/streamlit_export_501")
export_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(dataset_raw / "dataset.json", export_dir / "dataset.json")
print("[1/4] Copied dataset.json")

trainer_folders = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
if not trainer_folders:
    print("ERROR: No trained model found!")
else:
    trainer_folder  = trainer_folders[0]
    export_trainer  = export_dir / trainer_folder.name
    export_trainer.mkdir(exist_ok=True)
    for fname in ["dataset.json", "dataset_fingerprint.json", "plans.json"]:
        src = trainer_folder / fname
        if src.exists(): shutil.copy2(src, export_trainer / fname)
    for fold_dir in sorted(trainer_folder.glob("fold_*")):
        dst_fold = export_trainer / fold_dir.name
        if dst_fold.exists(): shutil.rmtree(dst_fold)
        shutil.copytree(fold_dir, dst_fold)
        print(f"  Copied {fold_dir.name}")
    print(f"[2/4] Exported trainer: {trainer_folder.name}")
    pp = results_dir / "postprocessing.pkl"
    if pp.exists():
        shutil.copy2(pp, export_dir / "postprocessing.pkl")
        print("[3/4] Copied postprocessing.pkl")
    else:
        print("[3/4] No postprocessing.pkl (run find_best_configuration first)")
    (export_dir / "README.txt").write_text(
        "nnUNetV2 Model Export — Dataset501_KidneyStones\n"
        "Config: 3d_fullres\n"
        "Folds : fold_0 to fold_4\n"
    )
    print("[4/4] Created README.txt")

print(f"\nExport: {export_dir}")

In [ ]:
import shutil
from pathlib import Path

zip_path   = Path(RESULTS_DRIVE_PATH) / "streamlit_model_export_501.zip"
export_dir = Path("/content/streamlit_export_501")

shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", export_dir)
size_mb = zip_path.stat().st_size / 1e6
print(f"Zip saved: {zip_path}")
print(f"Size: {size_mb:.1f} MB")